# 47 — FAISS Vector Search
**Goal:** Use FAISS for fast semantic similarity search over resume embeddings.

## 1. FAISS Basics

In [ ]:
print('''FAISS = Facebook AI Similarity Search
Purpose: Fast nearest-neighbor search in high-dimensional spaces
Key: Build an index of resume embeddings, query with JD embedding

Index types:
- IndexFlatL2  -> exact L2 distance (brute force, most accurate)
- IndexFlatIP  -> inner product / cosine similarity
- IndexIVFFlat -> approximate (faster, slightly less accurate)
- IndexHNSWFlat -> graph-based (fast, good accuracy)

For <100K resumes: IndexFlatIP is fine (exact search).''')

## 2. Building a FAISS Index

In [ ]:
import numpy as np
import faiss

d = 384  # dimension (all-MiniLM-L6-v2)
index = faiss.IndexFlatIP(d)
print(f"Empty index: {index.ntotal} vectors, dimension {index.d}")

# Simulate resume embeddings
np.random.seed(42)
resume_embeddings = np.random.randn(10, d).astype(np.float32)
# Normalize for cosine similarity
faiss.normalize_L2(resume_embeddings)
index.add(resume_embeddings)
print(f"After adding 10 resumes: {index.ntotal} vectors")

## 3. Querying with a JD

In [ ]:
# Query: job description embedding
jd_embedding = np.random.randn(1, d).astype(np.float32)
faiss.normalize_L2(jd_embedding)

# Search top-3 most similar resumes
k = 3
scores, indices = index.search(jd_embedding, k)
print("Top matches:")
for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
    print(f"  Rank {i+1}: Resume {idx} (score: {score:.4f})")

## 4. Index Persistence

In [ ]:
faiss.write_index(index, "/tmp/resume_index.faiss")
loaded_index = faiss.read_index("/tmp/resume_index.faiss")
print(f"Saved and reloaded: {loaded_index.ntotal} vectors, d={loaded_index.d}")

# IVF for larger scale
nlist = 2
quantizer = faiss.IndexFlatIP(d)
ivf_index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
ivf_index.train(resume_embeddings)
ivf_index.add(resume_embeddings)
ivf_index.nprobe = 1  # number of probes at search time
scores_ivf, indices_ivf = ivf_index.search(jd_embedding, k)
print(f"IVF results: indices={indices_ivf[0]}, scores={scores_ivf[0].round(4)}")

## Summary: FAISS enables fast resume retrieval from large candidate pools. IVF scales to millions.